# Ungraded Lab: Mask R-CNN Image Segmentation Demo (PyTorch)

In this lab, you will see how to use a [Mask R-CNN](https://arxiv.org/abs/1703.06870) model from torchvision for object detection and instance segmentation. This means that aside from the bounding boxes, the model is also able to predict segmentation masks for each instance of a class in the image. You have already encountered most of the commands here when you worked with the torchvision detection models in Week 2 and you will see how you can use it with instance segmentation models. Let's begin!

> This notebook is a PyTorch port of the original TensorFlow Hub / Object Detection API lab. There is nothing to install or compile: the pre-trained `maskrcnn_resnet50_fpn_v2` model (trained on COCO 2017) ships with torchvision, and the visualization utilities of the Object Detection API are replaced by `torchvision.utils`.

## Import libraries

In [ ]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks

import os
import matplotlib
import matplotlib.pyplot as plt

import numpy as np
from io import BytesIO
from PIL import Image
from urllib.request import urlopen

from urllib import request

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

%matplotlib inline

## Utilities

For convenience, you will use a function to convert an image to a numpy array. You can pass in a relative path to an image (e.g. to a local directory) or a URL. You can see this in the `TEST_IMAGES` dictionary below. Some paths point to the test images that come with the TensorFlow Object Detection API (e.g. `Beach`, fetched from GitHub) while others are URLs that point to images online (e.g. `Street`).

In [ ]:
def load_image_into_numpy_array(path):
  """Load an image from file into a numpy array.

  Puts image into numpy array to feed into the model.
  Note that by convention we put it into a numpy array with shape
  (height, width, channels), where channels=3 for RGB.

  Args:
    path: the file path to the image

  Returns:
    uint8 numpy array with shape (1, img_height, img_width, 3)
  """
  image = None
  if(path.startswith('http')):
    # Added User-Agent header to prevent 403 Forbidden error
    req = urlopen(request.Request(path, headers={'User-Agent': 'Mozilla/5.0'}))
    image_data = req.read()
    image_data = BytesIO(image_data)
    image = Image.open(image_data)
  else:
    image = Image.open(path)

  image = image.convert("RGB")
  (im_width, im_height) = (image.size)
  return np.array(image.getdata()).reshape(
      (1, im_height, im_width, 3)).astype(np.uint8)


# dictionary with image tags as keys, and image paths as values
TEST_IMAGES = {
  'Beach' : 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image2.jpg',
  'Dogs' : 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image1.jpg',
  # By Américo Toledano, Source: https://commons.wikimedia.org/wiki/File:Biblioteca_Maim%C3%B3nides,_Campus_Universitario_de_Rabanales_007.jpg
  'Phones' : 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/0d/Biblioteca_Maim%C3%B3nides%2C_Campus_Universitario_de_Rabanales_007.jpg/960px-Biblioteca_Maim%C3%B3nides%2C_Campus_Universitario_de_Rabanales_007.jpg',
  # By 663highland, Source: https://commons.wikimedia.org/wiki/File:Kitano_Street_Kobe01s5s4110.jpg
  'Street' : 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/3840px-Kitano_Street_Kobe01s5s4110.jpg'
}

## Load the Model

torchvision provides a Mask R-CNN model with a ResNet-50 FPN backbone. You can read about the details [here](https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html). Let's first load the model and see how to use it for inference in the next section.

In [ ]:
model_display_name = 'Mask R-CNN ResNet-50 FPN v2'
model_weights = MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1

print('Selected model:'+ model_display_name)
print('Model weights: {}'.format(model_weights.url))

In [ ]:
# The weights (about 170 MB) are downloaded the first time this runs
print('loading model...')
model = maskrcnn_resnet50_fpn_v2(weights=model_weights).to(device).eval()
print('model loaded!')

## Inference

You will use the model you just loaded to do instance segmentation on an image. First, choose one of the test images you specified earlier and load it into a numpy array.

In [ ]:
# Choose one and use as key for TEST_IMAGES below:
# ['Beach', 'Street', 'Dogs','Phones']

image_path = TEST_IMAGES['Beach']

image_np = load_image_into_numpy_array(image_path)

plt.figure(figsize=(24,32))
plt.imshow(image_np[0])
plt.show()

You can run inference by passing a list of image tensors to the model. Each image is a `(3, height, width)` float tensor with values in `[0, 1]`, and the model returns one dictionary of results per image. These are described in the `Outputs` section of the [documentation](https://pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn_v2.html): `boxes` (in pixels, `[xmin, ymin, xmax, ymax]`), `labels`, `scores` and `masks`.

In [ ]:
# convert the (1, height, width, 3) uint8 array to a (3, height, width) float tensor in [0, 1]
image_tensor = torch.from_numpy(image_np[0]).permute(2, 0, 1).float() / 255.0

# run inference. this can take a minute on a CPU.
with torch.no_grad():
  results = model([image_tensor.to(device)])[0]

# output values are tensors and we only need the numpy()
# parameter when we visualize the results
result = {key:value.cpu().numpy() for key,value in results.items()}

# print the keys and shapes
for key in result.keys():
  print(key, result[key].shape)

## Visualizing the results

You can now plot the results on the original image. First, you need to create the `category_index` dictionary that will contain the class IDs and names. The model was trained on the [COCO2017 dataset](https://cocodataset.org/) and torchvision stores the class names in the weights' metadata (`weights.meta["categories"]`). You can build the same dictionary format that the Object Detection API used.

In [ ]:
category_index = {idx: {'id': idx, 'name': name} for idx, name in enumerate(model_weights.meta["categories"])}

# sample output
print(category_index[1])
print(category_index[2])
print(category_index[4])

Next, you will preprocess the masks then finally plot the results.

* The result dictionary contains a `masks` key containing a segmentation mask for each box. Unlike the Object Detection API, torchvision already *reframes* the masks to the full image size, so each mask has the shape `(1, height, width)` with values between 0 and 1.
* You will also select mask pixel values that are above a certain threshold. We picked a value of `0.6` but feel free to modify this and see what results you will get. If you pick something lower, then you'll most likely notice mask pixels that are outside the object.
* You can use torchvision's `draw_segmentation_masks()` and `draw_bounding_boxes()` to plot the results on the image (the equivalent of `visualize_boxes_and_labels_on_image_array()`).

You can see how all these are handled in the code below.

In [ ]:
# Handle models with masks:
label_id_offset = 0
image_np_with_mask = image_np.copy()

# keep only the detections with a score above the threshold
min_score_thresh = .70
max_boxes_to_draw = 100
keep = np.where(result['scores'] >= min_score_thresh)[0][:max_boxes_to_draw]

boxes = torch.from_numpy(result['boxes'][keep])
scores = result['scores'][keep]
classes = (result['labels'][keep] + label_id_offset).astype(int)
labels = [f"{category_index[c]['name']}: {int(100 * s)}%" for c, s in zip(classes, scores)]

if 'masks' in result:

  # filter mask pixel values that are above a specified threshold
  detection_masks = torch.from_numpy(result['masks'][keep][:, 0] > 0.6)

  result['detection_masks_reframed'] = detection_masks.numpy()

# overlay labeled boxes and segmentation masks on the image
image_uint8 = torch.from_numpy(image_np_with_mask[0]).permute(2, 0, 1)
colors = [matplotlib.colors.to_hex(plt.cm.tab20(c % 20)) for c in classes]
# a TrueType font is needed to draw the labels at a readable size; matplotlib ships DejaVu Sans
font_path = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data/fonts/ttf/DejaVuSans.ttf")
if 'masks' in result:
  image_uint8 = draw_segmentation_masks(image_uint8, detection_masks, alpha=0.4, colors=colors)
image_uint8 = draw_bounding_boxes(image_uint8, boxes, labels=labels, colors=colors, width=8,
                                  font=font_path, font_size=max(20, image_uint8.shape[1] // 40))
image_np_with_mask[0] = image_uint8.permute(1, 2, 0).numpy()

plt.figure(figsize=(24,32))
plt.imshow(image_np_with_mask[0])
plt.show()